# Modello Discriminativo XGBoost

In [1]:
import csv
import re
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        holdout = candidate / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_holdout.csv'
        if holdout.exists():
            return candidate
    raise FileNotFoundError('Impossibile trovare la radice del progetto.')


project_root = find_project_root()
holdout_path = project_root / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_ML' / 'heloc_ML_holdout.csv'
imputated_root = project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_ML'
output_dir = project_root / 'data' / 'processed' / 'Fase3' / 'Results'
TARGET_COL = 'RiskPerformance'

print(f'Project root : {project_root}')
print(f'Holdout      : {holdout_path} (esiste: {holdout_path.exists()})')
print(f'Imputated dir: {imputated_root}')
print(f'Output dir   : {output_dir}')


Project root : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project
Holdout      : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase2/SplitDataset/Split_ML/heloc_ML_holdout.csv (esiste: True)
Imputated dir: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Imputated_ML
Output dir   : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results


In [2]:
def load_csv_as_df(path: Path) -> pd.DataFrame:
    """Legge un CSV e converte le colonne numeriche, lasciando NaN per i campi vuoti."""
    df = pd.read_csv(path)
    return df


def prepare_Xy(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """Separa features e target; converte il target in 0/1."""
    y = (df[TARGET_COL] == 'Good').astype(int)   # Good=1, Bad=0
    X = df.drop(columns=[TARGET_COL])
    # Converte eventuali colonne object residue in float (i boolean _never/_no_trades)
    for col in X.select_dtypes(include='object').columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    return X, y


def parse_train_filename(name: str) -> tuple[str, str, str]:
    """Estrae (dataset, strategia, pct) dal nome file discrirminative_train."""
    m = re.match(r'^(.+?)_discrirminative_train_([A-Z]+)_(\d+)\.csv$', name)
    if not m:
        raise ValueError(f'Nome file non riconosciuto: {name}')
    return m.group(1), m.group(2), m.group(3)


# Carica holdout una sola volta
df_holdout = load_csv_as_df(holdout_path)
X_holdout, y_holdout = prepare_Xy(df_holdout)
print(f'Holdout: {X_holdout.shape[0]} righe, {X_holdout.shape[1]} feature')
print(f'Distribuzione target holdout: Good={y_holdout.sum()}, Bad={(y_holdout==0).sum()}')

Holdout: 2959 righe, 34 feature
Distribuzione target holdout: Good=1420, Bad=1539


In [ ]:
# Iperparametri XGBoost
XGBOOST_PARAMS = dict(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

results = []
imputation_methods = ['Mediana', 'MICE']

for method in imputation_methods:
    train_dir = imputated_root / method
    train_files = sorted(train_dir.glob('*_discrirminative_train_*.csv'))

    print(f'\n=== {method} — {len(train_files)} file ===')

    for train_path in train_files:
        dataset, strategia, pct = parse_train_filename(train_path.name)

        # Carica train 
        df_train = load_csv_as_df(train_path)
        X_train, y_train = prepare_Xy(df_train)

        # Addestra XGBoost sul train
        model = XGBClassifier(**XGBOOST_PARAMS)
        model.fit(X_train, y_train)

        # Valuta sul holdout pulito
        y_pred = model.predict(X_holdout)

        acc   = accuracy_score(y_holdout, y_pred)
        f1    = f1_score(y_holdout, y_pred, pos_label=1)

        row = {
            'imputation_method': method,
            'dataset'          : dataset,
            'missing_strategy' : strategia,
            'missing_pct'      : int(pct),
            'train_file'       : train_path.name,
            'holdout_file'     : holdout_path.name,
            'train_rows'       : len(X_train),
            'holdout_rows'     : len(X_holdout),
            'accuracy'         : round(acc, 6),
            'f1_score'         : round(f1, 6),
        }
        results.append(row)
        print(f'  {strategia:4s} {pct:>2s}% | Acc={acc:.4f} F1={f1:.4f}')

print(f'\nTotale esperimenti: {len(results)}')


=== Mediana — 9 file ===
  MAR  10% | Acc=0.7036 F1=0.7006
  MAR  25% | Acc=0.7100 F1=0.6925
  MAR  40% | Acc=0.6830 F1=0.6761
  MCAR 10% | Acc=0.7141 F1=0.7058
  MCAR 25% | Acc=0.6921 F1=0.6571
  MCAR 40% | Acc=0.6634 F1=0.5938
  MNAR 10% | Acc=0.7138 F1=0.7078
  MNAR 25% | Acc=0.6911 F1=0.6859
  MNAR 40% | Acc=0.6887 F1=0.6860

=== MICE — 9 file ===
  MAR  10% | Acc=0.7077 F1=0.6987
  MAR  25% | Acc=0.7127 F1=0.6977
  MAR  40% | Acc=0.7104 F1=0.7017
  MCAR 10% | Acc=0.7165 F1=0.6959
  MCAR 25% | Acc=0.7121 F1=0.6804
  MCAR 40% | Acc=0.7117 F1=0.6908
  MNAR 10% | Acc=0.7276 F1=0.7103
  MNAR 25% | Acc=0.6914 F1=0.6617
  MNAR 40% | Acc=0.7083 F1=0.7041

Totale esperimenti: 18


In [4]:
# Salva i risultati in CSV
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / 'discriminative_results_ML_xgboost.csv'

fieldnames = [
    'imputation_method', 'dataset', 'missing_strategy', 'missing_pct',
    'train_file', 'holdout_file', 'train_rows', 'holdout_rows',
    'accuracy', 'f1_score',
]
with report_path.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f'Report salvato: {report_path}')

# Mostra tabella riassuntiva
df_results = pd.DataFrame(results)
df_results.sort_values(['imputation_method', 'missing_strategy', 'missing_pct'])


Report salvato: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results/discriminative_results_ML_xgboost.csv


,imputation_method,dataset,missing_strategy,missing_pct,train_file,holdout_file,train_rows,holdout_rows,accuracy,f1_score
9,MICE,heloc_ML,MAR,10,heloc_ML_discrirminative_train_MAR_10.csv,heloc_ML_holdout.csv,2958,2959,0.707672,0.698711
10,MICE,heloc_ML,MAR,25,heloc_ML_discrirminative_train_MAR_25.csv,heloc_ML_holdout.csv,2958,2959,0.712741,0.697724
11,MICE,heloc_ML,MAR,40,heloc_ML_discrirminative_train_MAR_40.csv,heloc_ML_holdout.csv,2958,2959,0.710375,0.701706
12,MICE,heloc_ML,MCAR,10,heloc_ML_discrirminative_train_MCAR_10.csv,heloc_ML_holdout.csv,2958,2959,0.716458,0.695904
13,MICE,heloc_ML,MCAR,25,heloc_ML_discrirminative_train_MCAR_25.csv,heloc_ML_holdout.csv,2958,2959,0.712065,0.680420
14,MICE,heloc_ML,MCAR,40,heloc_ML_discrirminative_train_MCAR_40.csv,heloc_ML_holdout.csv,2958,2959,0.711727,0.690830
15,MICE,heloc_ML,MNAR,10,heloc_ML_discrirminative_train_MNAR_10.csv,heloc_ML_holdout.csv,2958,2959,0.727611,0.710280
16,MICE,heloc_ML,MNAR,25,heloc_ML_discrirminative_train_MNAR_25.csv,heloc_ML_holdout.csv,2958,2959,0.691450,0.661727
17,MICE,heloc_ML,MNAR,40,heloc_ML_discrirminative_train_MNAR_40.csv,heloc_ML_holdout.csv,2958,2959,0.708347,0.704148
0,Mediana,heloc_ML,MAR,10,heloc_ML_discrirminative_train_MAR_10.csv,heloc_ML_holdout.csv,2958,2959,0.703616,0.700580
